In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns 
from sklearn.preprocessing import StandardScaler

print("Everything imported!")

In [ ]:
df = pd.read_csv("Titanic-Dataset.csv")
print("Local file successfully loaded into Pandas")

In [ ]:
df.info()

In [ ]:
# 1x2 layout for raw data diagnostics
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))
fig.suptitle('Raw Dataset Integrity & Missing Data Diagnosis', fontsize=18, fontweight='bold')

# Plot 1: Visual Map of Missing Values (Heatmap)
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis', ax=axes[0])
axes[0].set_title('Missing Data Matrix (Yellow lines = Missing Holes)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Dataset Features / Columns')
axes[0].set_ylabel('Passenger Rows (1 to 891)')

# Plot 2: Exact Missing Count Breakdown (Bar Chart)
missing_counts = df.isnull().sum()
missing_only = missing_counts[missing_counts > 0].sort_values(ascending=False)

if not missing_only.empty:
    sns.barplot(x=missing_only.index, y=missing_only.values, palette='Reds_r', ax=axes[1])
    for i, val in enumerate(missing_only.values):
        axes[1].text(i, val + 10, f"{val}\n({(val/len(df))*100:.1f}%)", ha='center', fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'No Missing Values Found!', ha='center', va='center', fontsize=14)

axes[1].set_title('Exact Count & Percentage of Missing Values', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Columns with Missing Data')
axes[1].set_ylabel('Number of Missing Rows')
axes[1].set_ylim(0, 891)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Filled missing Age values with the median age of all passengers
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)

# 2. Filled missing Embarked values with the most common port (Mode)
most_frequent_port = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(most_frequent_port)

# 3. Dropped the Cabin column entirely because too much data is missing
if 'Cabin' in df.columns:
    df = df.drop(columns=['Cabin'])

print("Remaining Missing Values per Column:")
print(df.isnull().sum())

In [ ]:
df.info()

In [ ]:
# 1. Create a figure to look at the ORIGINAL, unscaled Age and Fare distributions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(y=df['Age'], color='skyblue')
plt.title('Age Distribution (Actual Values - Pre-Clean)')
plt.ylabel('Age (Years)')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['Fare'], color='salmon')
plt.title('Fare Distribution (Actual Values - Pre-Clean)')
plt.ylabel('Fare Price')

plt.tight_layout()
plt.show()

# 2. Use the IQR Method to filter out the extreme outliers on raw features
print(f"Rows before outlier removal: {len(df)}")

def remove_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return dataframe[(dataframe[column] >= lower_bound) & (dataframe[column] <= upper_bound)]

# Clean outliers dynamically out of both continuous columns
df_cleaned = remove_outliers_iqr(df, 'Age')
df_cleaned = remove_outliers_iqr(df_cleaned, 'Fare')

print(f"Rows after outlier removal: {len(df_cleaned)}")

# Plot distributions after cleaning to showcase improvement
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(y=df_cleaned['Age'], color='lightgreen')
plt.title('Age Distribution (Actual Values - Post-Clean)')
plt.ylabel('Age (Years)')

plt.subplot(1, 2, 2)
sns.boxplot(y=df_cleaned['Fare'], color='lightpink')
plt.title('Fare Distribution (Actual Values - Post-Clean)')
plt.ylabel('Fare Price')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Binary Encode the 'Sex' column
df_cleaned['Sex'] = df_cleaned['Sex'].map({'male': 1, 'female': 0})

# 2. One-Hot Encode the 'Embarked' column
df_cleaned = pd.get_dummies(df_cleaned, columns=['Embarked'], prefix='Embarked', dtype=int)

# 3. Drop columns that are text-heavy and descriptive
columns_to_drop = ['Name', 'Ticket', 'PassengerId']
df_cleaned = df_cleaned.drop(columns=[col for col in columns_to_drop if col in df_cleaned.columns])

print("Categorical features converted to numerical formats successfully!")
print("\nNew columns present in the dataset:")
print(df_cleaned.columns.tolist())
df_cleaned.head()

In [ ]:
# 1. Initialize the StandardScaler
scaler = StandardScaler()

# 2. Select continuous numerical columns
columns_to_scale = ['Age', 'Fare']

# 3. Fit and transform the OUTLIER-FREE dataset
df_cleaned[columns_to_scale] = scaler.fit_transform(df_cleaned[columns_to_scale])


df_cleaned[columns_to_scale] = df_cleaned[columns_to_scale].round(4)

print("Continuous numerical features successfully standardized!")
print("\nSample values from the scaled columns (Perfect 4-point decimals centered around 0):")
print(df_cleaned[['Age', 'Fare']].head())

In [ ]:
df.info()

In [ ]:
# Setting up a 1x2 layout for post-cleaning diagnostics
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))
fig.suptitle('Post-Preprocessing Dataset Integrity & Health Check', fontsize=18, fontweight='bold')

# Plot 1: Visual Map of Missing Values (Heatmap after cleaning)
sns.heatmap(df_cleaned.isnull(), yticklabels=False, cbar=False, cmap='viridis', ax=axes[0])
axes[0].set_title('Cleaned Data Matrix (Solid uniform validation)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Dataset Features / Columns')
axes[0].set_ylabel('Passenger Rows Remaining')

# Plot 2: Missing Count Breakdown
cleaned_missing_counts = df_cleaned.isnull().sum()
cleaned_missing_only = cleaned_missing_counts[cleaned_missing_counts > 0].sort_values(ascending=False)

if cleaned_missing_only.empty:
    axes[1].text(0.5, 0.5, 'Zero Missing Values Found!', 
                 ha='center', va='center', fontsize=16, fontweight='bold', color='green')
    axes[1].grid(False) 
else:
    sns.barplot(x=cleaned_missing_only.index, y=cleaned_missing_only.values, palette='Reds_r', ax=axes[1])

axes[1].set_title('Exact Count & Percentage of Missing Values', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Columns with Missing Data')
axes[1].set_ylabel('Number of Missing Rows')
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("\nTASK 1 COMPLETE! Final dataset diagnostic look:")
df_cleaned.info()

In [ ]:
# Remove comments to Save the dataframe to a new CSV file.
df_cleaned.to_csv('cleaned_titanic.csv', index=False)

print("Your cleaned data has been saved as a new file: 'cleaned_titanic.csv'!")